# Epsilon Fund — Strategy Testing
---

In [21]:
import pandas as pd
import numpy as np
import sys

# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research'  # ← change to yours
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + '/infrastructure/data')
sys.path.append(ROOT + '/infrastructure/backtester')

from binance_client import get_binance_client, get_data, get_multiple_data
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: `365` (1y) · `730` (2y) · `1825` (5y) · `2555` (7y, recommended minimum)

In [22]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1d'
LOOKBACK = 1200

# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)

print(f'{SYMBOL} | {INTERVAL} | {df.index[0].date()} -> {df.index[-1].date()} | {len(df)} rows')
df.tail(3)

BTCUSDT | 1d | 2022-12-05 -> 2026-03-18 | 1200 rows


,Open,High,Low,Close,Volume
Time,,,,,
2026-03-16,72815.25,74909.08,72270.41,74884.67,28409.53019
2026-03-17,74884.67,76000.00,73399.19,73909.36,24521.26636
2026-03-18,73909.37,74672.34,70857.21,71324.44,15902.81194


---
## Strategy

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> Signals are shifted 1 bar by the engine — no need to shift manually.

In [23]:
strategy_df = df.copy()

# ── Strategy logic ─────────────────────────────────────────────────────────────

# Indicators
strategy_df['EMA_14'] = strategy_df['Close'].ewm(span=14, adjust=False).mean()
strategy_df['Swing_High_7'] = strategy_df['High'].rolling(window=7).max()
strategy_df['Swing_Low_7']  = strategy_df['Low'].rolling(window=7).min()
strategy_df['Swing_High_3'] = strategy_df['High'].rolling(window=5).max()
strategy_df['Swing_Low_3']  = strategy_df['Low'].rolling(window=5).min()

# ATR
def calculate_atr(df, period):
    high_low    = df['High'] - df['Low']
    high_close  = np.abs(df['High'] - df['Close'].shift())
    low_close   = np.abs(df['Low']  - df['Close'].shift())
    true_range  = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return true_range.rolling(window=period).mean()

strategy_df['ATR_21']   = calculate_atr(strategy_df, 21)
strategy_df['ATR_roc'] = strategy_df['ATR_21'].pct_change(periods=14)

# ADX
def calculate_adx(high, low, close, period=14):
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low  - close.shift())
    tr  = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    up_move   = high - high.shift()
    down_move = low.shift() - low

    plus_dm  = np.where((up_move > down_move) & (up_move > 0),   up_move,   0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)

    atr      = tr.ewm(alpha=1/period, adjust=False).mean()
    plus_dm  = pd.Series(plus_dm,  index=close.index).ewm(alpha=1/period, adjust=False).mean()
    minus_dm = pd.Series(minus_dm, index=close.index).ewm(alpha=1/period, adjust=False).mean()

    plus_di  = 100 * (plus_dm  / atr)
    minus_di = 100 * (minus_dm / atr)

    dx  = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
    adx = dx.rolling(period).mean()
    return adx

strategy_df['ADX_14'] = calculate_adx(strategy_df['High'], strategy_df['Low'], strategy_df['Close'], 14)

# Caution flags
strategy_df['Caution_Long'] = (
    ((strategy_df['Swing_High_7'] - strategy_df['Low']) > (1.5 * strategy_df['ATR_21'])) |
    (strategy_df['Close'] < strategy_df['EMA_14'])
)
strategy_df['Caution_Short'] = (
    ((strategy_df['High'] - strategy_df['Swing_Low_7']) > (1.5 * strategy_df['ATR_21'])) |
    (strategy_df['Close'] > strategy_df['EMA_14']) |
    (strategy_df['ATR_roc'] > 0.4)
)

# Entry signals
strategy_df['Entry_Long'] = (
    (strategy_df['Close'] > strategy_df['EMA_14']) &
    ((~strategy_df['Caution_Long']) | (strategy_df['ADX_14'] > 60))
)
strategy_df['Entry_Short'] = (
    (strategy_df['Close'] < strategy_df['EMA_14']) &
    ((~strategy_df['Caution_Short']) | (strategy_df['ADX_14'] > 60)) &
    (strategy_df['ATR_roc'] < 0.21)
)

# Position loop
strategy_df['position']    = 0
strategy_df['Stop_Loss']   = np.nan
strategy_df['Entry_Price'] = np.nan
in_position = 0
stop_loss = entry_price = 0

for i in range(20, len(strategy_df)):
    idx      = strategy_df.index[i]
    prev_idx = strategy_df.index[i - 1]
    curr     = strategy_df.loc[idx]
    prev     = strategy_df.loc[prev_idx]

    # Stop multiplier
    if in_position == 1:
        if curr['Caution_Long']:    stop_multiplier = 0.4
        else:                       stop_multiplier = 1.0
    elif in_position == -1:
            if curr['Caution_Long'] and curr['Caution_Short']: stop_multiplier = 0.6
            elif curr['Caution_Short']: stop_multiplier = 0.4
            elif curr['Caution_Long']: stop_multiplier = 0.5
            else:                       stop_multiplier = 1.0
    else:
        if prev['Entry_Long']:
            if curr['Caution_Long'] and curr['Caution_Short']: stop_multiplier = 0.7
            elif curr['Caution_Long']: stop_multiplier = 0.4
            elif curr['Caution_Short']: stop_multiplier = 0.4
            else:                       stop_multiplier = 1.0
        elif prev['Entry_Short']:
            if curr['Caution_Long'] and curr['Caution_Short']: stop_multiplier = 0.7
            elif curr['Caution_Long'] or curr['Caution_Short']: stop_multiplier = 0.4
            else:                       stop_multiplier = 1.0
        else:
            stop_multiplier = 1.0
    
    stop_distance = curr['ATR_21'] * stop_multiplier * 1.

    # Long management
    if in_position == 1:
        if prev['Close'] < stop_loss:
            in_position = 0
            strategy_df.loc[idx, 'position'] = 0
            continue
        stop_loss = max(stop_loss, curr['Swing_High_3'] - stop_distance)
        strategy_df.loc[idx, ['position', 'Stop_Loss']] = [1, stop_loss]

    # Short management
    elif in_position == -1:
        if prev['Close'] > stop_loss:
            in_position = 0
            strategy_df.loc[idx, 'position'] = 0
            continue
        stop_loss = min(stop_loss, curr['Swing_Low_3'] + stop_distance)
        strategy_df.loc[idx, ['position', 'Stop_Loss']] = [-1, stop_loss]

    # Entry when flat
    else:
        if prev['Entry_Long']:
            stop_loss   = curr['Swing_High_3'] - stop_distance
            in_position = 1
            strategy_df.loc[idx, ['position', 'Entry_Price', 'Stop_Loss']] = [1, entry_price, stop_loss]

        elif prev['Entry_Short']:
            stop_loss   = curr['Swing_Low_3'] + stop_distance
            in_position = -1
            strategy_df.loc[idx, ['position', 'Entry_Price', 'Stop_Loss']] = [-1, entry_price, stop_loss]


# ──────────────────────────────────────────────────────────────────────────────

indicator_cols = ['EMA_14', 'ATR_21', 'ADX_14', 'Swing_High_7']
strategy_df.dropna(subset=indicator_cols, inplace=True)

# Fill the position-related NaNs explicitly
strategy_df['position']    = strategy_df['position'].fillna(0).astype(int)
strategy_df['Stop_Loss']   = strategy_df['Stop_Loss'].fillna(0)
strategy_df['Entry_Price'] = strategy_df['Entry_Price'].fillna(0)

strategy_df['position'].value_counts()

position
 0    657
 1    340
-1    183
Name: count, dtype: int64

---
## Backtest

| Parameter | Options | Default |
|---|---|---|
| `cost` | Cost per trade as decimal — `0.001` = 0.1% | `0.0` |
| `show_plot` | `True` / `False` | `True` |
| `save_html` | Filename string or `None` | `None` |
| `show_trades` | Overlay entry/exit markers on price chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | same asset |

In [25]:
results = backtest(
    data           = strategy_df,
    cost           = 0.001,
    show_plot      = True,
    save_html      = "moneyin",
    show_trades    = True,
    benchmark_data = None
)

print(f"Return        {results['total_return']*100:>8.2f}%")
print(f"Sharpe        {results['sharpe_ratio']:>8.2f}")
print(f"Max Drawdown  {results['max_drawdown']*100:>8.2f}%")
print(f"Profit Factor {results['profit_factor']:>8.2f}")
print(f"Win Rate      {results['win_rate']*100:>8.2f}%")
print(f"# Trades      {results['num_trades']:>8}")

✓ Saved interactive chart to: moneyin


✓ Saved trade chart to: moneyin


Return          197.66%
Sharpe            1.26
Max Drawdown    -28.19%
Profit Factor     1.75
Win Rate         55.16%
# Trades           310
